In [1]:

import pandas as pd
from pathlib import Path
from IPython.display import display, HTML

# =========================================================
# 1. LER O ARQUIVO PARQUET
# =========================================================

arquivo = Path("07_PENSE2024_IDADE_13_A_17.parquet")

df = pd.read_parquet(arquivo, engine="pyarrow")

print(f"Arquivo: {arquivo}")
print(f"Linhas: {len(df):,}")
print(f"Colunas: {len(df.columns)}")

if len(df.columns) != 41:
    print("ATENÇÃO: o arquivo não contém 41 colunas.")

# =========================================================
# 2. CLASSIFICAÇÕES INICIAIS
# =========================================================

nominais = {
    "REGIAO", "UF", "CAPITAL", "B01001A", "B01002",
    "B01006", "B01007", "B01015B", "B01016",
    "B05002A", "B07013", "B07009", "B07016",
    "B12009", "B09016A1", "B09016A2",
    "E01P8505", "E01P8510", "E01P117",
    "E01P11801", "E01P11802"
}

ordinais = {
    "IDADE_AGREG",
    "B07002", "B07004", "B07005", "B07006",
    "B07007B", "B07011A", "B07012A",
    "B12003", "B12004", "B12005", "B12006",
    "B12007", "B12008", "B03010C",
    "B09003A", "B09010A1"
}

quantitativas_discretas = {
    "B03006B", "B09001", "B09002"
}

binarias = {
    "CAPITAL", "B01001A", "B01006", "B01007",
    "B01015B", "B01016", "B05002A", "B07013",
    "B07009", "B07016", "B12009",
    "B09016A1", "B09016A2",
    "E01P8505", "E01P8510",
    "E01P11801", "E01P11802"
}

# =========================================================
# 3. ALTERAÇÕES MANUAIS
# =========================================================
# Informe o tipo que você deseja para cada variável.
# Estas escolhas prevalecem sobre a classificação inicial.

classificacoes_manuais = {
    # Exemplos:
    # "REGIAO": "Qualitativa nominal",
    # "B03006B": "Quantitativa discreta",
    # "B03010C": "Qualitativa ordinal",
}

# Inclua aqui os códigos que você já revisou manualmente.
variaveis_revisadas = {
    # Exemplos:
    # "REGIAO",
    # "UF",
}

# =========================================================
# 4. EXTRAIR CÓDIGO E CLASSIFICAR
# =========================================================

def extrair_codigo(nome_coluna):
    return nome_coluna.split(" - ")[0].strip()

def classificar_variavel(codigo):
    if codigo in classificacoes_manuais:
        return classificacoes_manuais[codigo]

    if codigo in binarias:
        return "Qualitativa nominal binária"

    if codigo in nominais:
        return "Qualitativa nominal"

    if codigo in ordinais:
        return "Qualitativa ordinal"

    if codigo in quantitativas_discretas:
        return "Quantitativa discreta"

    return "Não classificada"

# =========================================================
# 5. GERAR RELATÓRIO
# =========================================================

linhas_relatorio = []

for coluna in df.columns:
    serie = df[coluna]
    codigo = extrair_codigo(coluna)

    valores = serie.dropna().unique()
    amostra = ", ".join(
        str(v) for v in sorted(valores, key=str)[:15]
    )

    if len(valores) > 15:
        amostra += ", ..."

    linhas_relatorio.append({
        "Código": codigo,
        "Variável": coluna,
        "Tipo Pandas": str(serie.dtype),
        "Classificação": classificar_variavel(codigo),
        "Não nulos": int(serie.notna().sum()),
        "Nulos": int(serie.isna().sum()),
        "Valores distintos": int(serie.nunique(dropna=True)),
        "Valores observados (amostra)": amostra,
        "Revisão manual": (
            "Realizado"
            if codigo in variaveis_revisadas
            else "Pendente"
        )
    })

relatorio = pd.DataFrame(linhas_relatorio)

# =========================================================
# 6. EXIBIR RELATÓRIO
# =========================================================

html = relatorio.to_html(
    index=False,
    escape=True,
    border=0,
    classes="classificacao-table"
)

display(HTML(f"""
<div style="max-height:500px; overflow:auto;
            border:1px solid #bbb; border-radius:6px;">
<style>
.classificacao-table {{
    border-collapse:collapse;
    width:100%;
    font-size:12px;
    color:#333;
    background:#e8e8e8;
}}
.classificacao-table th {{
    position:sticky;
    top:0;
    z-index:2;
    background:#d0d0d0;
    padding:8px;
    text-align:left;
    white-space:nowrap;
    border-bottom:2px solid #aaa;
}}
.classificacao-table td {{
    padding:7px;
    border-bottom:1px solid #ccc;
    background:#e8e8e8;
}}
.classificacao-table tr:nth-child(even) td {{
    background:#dedede;
}}
</style>
{html}
</div>
"""))

# =========================================================
# 7. RESUMO DAS CLASSIFICAÇÕES
# =========================================================

resumo = (
    relatorio["Classificação"]
    .value_counts()
    .rename_axis("Classificação")
    .reset_index(name="Quantidade")
)

print("\nRESUMO DAS CLASSIFICAÇÕES")
display(resumo)

nao_classificadas = relatorio[
    relatorio["Classificação"] == "Não classificada"
]

print(f"\nVariáveis não classificadas: {len(nao_classificadas)}")

if not nao_classificadas.empty:
    display(nao_classificadas[["Código", "Variável"]])

# =========================================================
# 8. SALVAR RELATÓRIOS EM CSV
# =========================================================

relatorio.to_csv(
    "08_RELATORIO_CLASSIFICACAO_VARIAVEIS.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

resumo.to_csv(
    "08_RESUMO_CLASSIFICACAO_VARIAVEIS.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print("\nRelatórios exportados:")
print("- 08_RELATORIO_CLASSIFICACAO_VARIAVEIS.csv")
print("- 08_RESUMO_CLASSIFICACAO_VARIAVEIS.csv")


Arquivo: 07_PENSE2024_IDADE_13_A_17.parquet
Linhas: 118,041
Colunas: 41


Código,Variável,Tipo Pandas,Classificação,Não nulos,Nulos,Valores distintos,Valores observados (amostra),Revisão manual
REGIAO,REGIAO - código de regiões,Int64,Qualitativa nominal,118041,0,5,"1, 2, 3, 4, 5",Pendente
UF,UF - código UF,Int64,Qualitativa nominal,118041,0,27,"11, 12, 13, 14, 15, 16, 17, 21, 22, 23, 24, 25, 26, 27, 28, ...",Pendente
CAPITAL,CAPITAL - indica se é município da capital,Int64,Qualitativa nominal binária,118041,0,2,"0, 1",Pendente
B01001A,B01001A - Qual é o seu sexo?,Int64,Qualitativa nominal binária,118041,0,2,"1, 2",Pendente
IDADE_AGREG,"IDADE_AGREG - IDADE DO ESTUDANTE, CATEGORIAS",Int64,Qualitativa ordinal,118041,0,2,"2, 3",Pendente
B01002,B01002 - Qual é a sua cor ou raça?,Int64,Qualitativa nominal,118041,0,6,"0, 1, 2, 3, 4, 5",Pendente
B01006,B01006 - Você mora com sua mãe?,Int64,Qualitativa nominal binária,118041,0,3,"0, 1, 3",Pendente
B01007,B01007 - Você mora com seu pai?,Int64,Qualitativa nominal binária,118041,0,3,"0, 1, 3",Pendente
B01015B,B01015B - Na sua casa tem computador ou notebook?,Int64,Qualitativa nominal binária,118041,0,3,"0, 1, 3",Pendente
B01016,B01016 - Você tem acesso à internet em sua casa?,Int64,Qualitativa nominal binária,118041,0,3,"0, 1, 3",Pendente



RESUMO DAS CLASSIFICAÇÕES


,Classificação,Quantidade
0,Qualitativa nominal binária,17
1,Qualitativa ordinal,17
2,Qualitativa nominal,4
3,Quantitativa discreta,3



Variáveis não classificadas: 0

Relatórios exportados:
- 08_RELATORIO_CLASSIFICACAO_VARIAVEIS.csv
- 08_RESUMO_CLASSIFICACAO_VARIAVEIS.csv
